# Pipeline SHAP — GFP Implementation Gap

**Descripción:** Calcula e interpreta los SHAP values del modelo XGBoost óptimo para cada modalidad.
Requiere haber corrido primero `02_modeling_pipeline.ipynb` (modelos `.joblib` ya entrenados).

**Flujo:**
1. Configuración de modalidad y modelo
2. Carga de data procesada + split train/test (idéntico al de modelado)
3. Carga del modelo XGB entrenado
4. Cálculo de SHAP values (TreeExplainer)
5. Tabla SHAP → Excel
6. Summary plot (beeswarm) → PNG

## 0. Configuración — cambiar aquí para cada modalidad

In [ ]:
# ============================================================
# CONFIGURACIÓN PRINCIPAL
# MODALIDAD : 'contrata' | 'ad' | 'arcc'
# SAMPLING  : 'o' | 's' | 'st' | 'nrs'
# TOP_N     : cuántas features mostrar en el gráfico
# ============================================================

MODALIDAD    = 'contrata'
SAMPLING     = 'o'
TOP_N        = 20
RANDOM_STATE = 2023
TEST_SIZE    = 0.2

PATH_DATA  = f'C:/15_GFP/data/processed/{MODALIDAD}/1_data_{MODALIDAD}.xlsx'
PATH_MODEL = f'C:/15_GFP/outputs/models/{MODALIDAD}/xgb_{SAMPLING}.joblib'
DIR_SHAP   = f'C:/15_GFP/outputs/shap/{MODALIDAD}'

print(f'Modalidad : {MODALIDAD.upper()}')
print(f'Modelo    : XGB {SAMPLING.upper()}')
print(f'Top N     : {TOP_N}')
print(f'Input     : {PATH_DATA}')
print(f'Modelo    : {PATH_MODEL}')
print(f'Output    : {DIR_SHAP}')

## 1. Importaciones

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

print(f'shap version: {shap.__version__}')

## 2. Carga de data y split train/test

In [ ]:
data = pd.read_excel(PATH_DATA, engine='openpyxl')

dep_var   = 'brecha_existente'
pred_vars = [col for col in data.columns if col != dep_var]

X = data[pred_vars]
y = data[dep_var]

# Split idéntico al de 02_modeling_pipeline.ipynb
x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y,
)

print(f'Train: {x_train.shape[0]:,} | Test: {x_test.shape[0]:,} | Features: {x_train.shape[1]}')
print(f'Distribución y_train: {dict(y_train.value_counts())}')

## 3. Carga del modelo XGB entrenado

In [ ]:
model = joblib.load(PATH_MODEL)
print(f'Modelo cargado: xgb_{SAMPLING} ({MODALIDAD.upper()})')
print(f'Tipo: {type(model).__name__}')

## 4. SHAP values (TreeExplainer)

In [ ]:
# TreeExplainer es el más eficiente y exacto para XGBoost/RF
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(x_train, approximate=False, check_additivity=False)

print(f'SHAP values shape: {shap_values.shape}')  # (n_obs, n_features)

## 5. Tabla SHAP → Excel

In [ ]:
shap_mean_abs = np.abs(shap_values).mean(axis=0)

shap_df = (
    pd.DataFrame({'feature': pred_vars, 'shap_mean_abs': shap_mean_abs})
    .sort_values('shap_mean_abs', ascending=False)
    .reset_index(drop=True)
)

os.makedirs(DIR_SHAP, exist_ok=True)
out_xlsx = f'{DIR_SHAP}/shap_values_xgb_{SAMPLING}.xlsx'
shap_df.to_excel(out_xlsx, index=False)
print(f'Tabla SHAP guardada: {out_xlsx}')

shap_df.head(TOP_N)

## 6. Summary plot (beeswarm) — Top N features

In [ ]:
# Seleccionar top N features por importancia SHAP media
top_features = shap_df['feature'].head(TOP_N).tolist()
top_idx      = [pred_vars.index(f) for f in top_features]

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 15,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
})

shap.summary_plot(
    shap_values[:, top_idx],
    x_train[top_features],
    feature_names=top_features,
    plot_size=(12, max(6, TOP_N * 0.4)),
    show=False,
    color_bar=True,
    cmap='coolwarm',
)

plt.title(f'SHAP Summary — XGB {SAMPLING.upper()} ({MODALIDAD.upper()}) — Top {TOP_N}', pad=12)
plt.tight_layout()

out_fig = f'{DIR_SHAP}/shap_summary_xgb_{SAMPLING}.png'
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {out_fig}')

## 7. [Opcional] Bar plot — importancia media absoluta

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

fig, ax = plt.subplots(figsize=(10, max(5, TOP_N * 0.35)))

plot_df = shap_df.head(TOP_N).iloc[::-1]  # invertir para que el más importante quede arriba
ax.barh(plot_df['feature'], plot_df['shap_mean_abs'], color='steelblue')
ax.set_xlabel('SHAP mean |value|')
ax.set_title(f'Feature Importance SHAP — XGB {SAMPLING.upper()} ({MODALIDAD.upper()}) — Top {TOP_N}')
plt.tight_layout()

out_bar = f'{DIR_SHAP}/shap_bar_xgb_{SAMPLING}.png'
plt.savefig(out_bar, dpi=300, bbox_inches='tight')
plt.show()
print(f'Bar plot guardado: {out_bar}')